In [1]:
import pandas as pd
import numpy as np
import os
import math
import matplotlib.pyplot as plt
import openpyxl
from DaySim import DaysimSummary
from Survey import DaysimSummary_Survey

In [2]:
pd.options.display.float_format = '{:,.1f}'.format

In [3]:
model = DaysimSummary()
survey = DaysimSummary_Survey()

runVehAvailability = True, loading data...
runWrkSchLocationChoice = True, loading data...
runTripMode = True, loading data...
runTourMode = True, loading data...
runTripDestination = True, loading data...
runTourDestination = True, loading data...
runTripTOD = True, loading data...
runTourTOD = True, loading data...
runDayPattern = True, loading data...
runVehAvailability = True, loading data...
runWrkSchLocationChoice = True, loading data...
runTripMode = True, loading data...
runTourMode = True, loading data...
runTripDestination = True, loading data...
runTourDestination = True, loading data...

Household Count by District:
   hhcounty  hh_count
0       5.0        21
runTripTOD = True, loading data...
runTourTOD = True, loading data...
runDayPattern = True, loading data...


In [4]:
import Survey

In [5]:
print(Survey.__file__)

c:\Users\USVA682771\OneDrive - WSP O365\Documents\Daysim_summaries\Survey.py


In [6]:
import inspect

In [7]:
print(inspect.isfunction(DaysimSummary_Survey.prep_trip_mode))

True


In [8]:
print(inspect.ismethod(DaysimSummary_Survey.prep_trip_mode))

False


In [9]:
print(type(DaysimSummary_Survey.prep_trip_mode))

<class 'function'>


### Helpers

In [10]:
PURPOSE_LABELS = {
    1: "Work",2: "School",3: "Escort",4: "Personal Business",5: "Shop",6: "Meal",7: "SocialRecreation",8: "Workbased"}

PPTYP_LABELS = {
    1: "FT Worker",2: "PT Worker",3: "Retired",4: "Nonworker",5: "Student",6: "Student_Age_16_17",7: "Student_Age_5_15", 8:"Under_Age_5"}

tables = {}

errors = {}

def col_pct(df):
    return df.div(df.sum(axis=0)) * 100

def row_pct(df):
    return df.div(df.sum(axis=1).to_numpy(),axis = 0) * 100

def compare_summaries(model, survey, pct_func=col_pct):
    m = pct_func(model)
    s = pct_func(survey).reindex(m.index, columns=m.columns, fill_value=0)
    d = s - m
    return pd.concat([m, s, d], keys=["Model %", "Survey %", "Difference (S-M)"])

def compare_and_store(key, model_df, survey_df, pct_func=col_pct):
    t = compare_summaries(model_df, survey_df, pct_func)
    tables[key] = t
    display(t)


In [11]:
class CompareDaysimSurvey:
    def __init__(self, model, survey):
        self.model = model
        self.survey = survey
        self.tables = {}

    def __getattr__(self, name):
        def wrapper(*args, pct_func=col_pct,**kwargs):
            key = "_".join([name] + [str(a) for a in args])[:31] 
            try:
                 m = getattr(self.model, name)(*args, **kwargs)
                 s = getattr(self.survey, name)(*args, **kwargs)
                 compare_and_store(key, m, s,pct_func=pct_func)
            except Exception as e:
                errors[key] = str(e)
                print(f'Error [{key}]: {e}')
        return wrapper

In [12]:
comparison  = CompareDaysimSurvey(model, survey)

# calibration summaries


In [13]:
survey.summary_vehavail_drivers_vehs()

hhvehcat,0,1,2,3,4
hh16cat,,,,,
1,"51,760.4","314,980.4","60,905.2","20,285.9","3,728.3"
2,"20,803.6","102,450.5","383,666.3","121,885.8","48,173.0"
3,"1,381.6","22,796.5","44,737.9","77,275.7","33,329.7"
4,"5,103.3","6,625.5","14,448.5","22,773.2","37,757.9"


In [14]:
comparison.summary_vehavail_drivers_vehs()

hhvehcat                     0    1    2    3    4
                 hh16cat                          
Model %          1        77.5 76.5 14.2  7.1  9.4
                 2        17.4 20.0 74.7 56.2 41.9
                 3         2.5  2.3  9.9 23.2 18.8
                 4         2.6  1.2  1.2 13.6 30.0
Survey %         1        65.5 70.5 12.1  8.4  3.0
                 2        26.3 22.9 76.2 50.3 39.2
                 3         1.7  5.1  8.9 31.9 27.1
                 4         6.5  1.5  2.9  9.4 30.7
Difference (S-M) 1       -12.0 -6.0 -2.1  1.3 -6.4
                 2         8.9  2.9  1.5 -5.8 -2.7
                 3        -0.8  2.8 -1.0  8.7  8.3
                 4         3.8  0.3  1.7 -4.2  0.7

no entries for survey while pwaudist is in the person input file and wfh is fixec

In [15]:
comparison.summary_wrkschloc_ft_nwfh_dist()

psexpfac
                 wrkdist3cat          
Model %          0-3.5 mi         17.7
                 3.5-10 mi        36.5
                 10+ mi           45.8
Survey %         0-3.5 mi          0.0
                 3.5-10 mi         0.0
                 10+ mi            0.0
Difference (S-M) 0-3.5 mi        -17.7
                 3.5-10 mi       -36.5
                 10+ mi          -45.8

In [16]:
survey.summary_wrkschloc_ft_nwfh_dist()

,psexpfac
wrkdist3cat,


no sfh

In [17]:
comparison.summary_wrkschloc_sch_dist()

schdist3cat              0-1 mi  1-5 mi  5+ mi
                 stutyp                       
Model %          Ch515     82.6    80.2   45.9
                 Stu16      4.4    12.2   12.8
                 UniStu    13.0     7.7   41.3
Survey %         Ch515      0.0     0.0    0.0
                 Stu16      0.0     0.0    0.0
                 UniStu     0.0     0.0    0.0
Difference (S-M) Ch515    -82.6   -80.2  -45.9
                 Stu16     -4.4   -12.2  -12.8
                 UniStu   -13.0    -7.7  -41.3

In [18]:
comparison.summary_day_pattern_purpose_rate("work")

psexpfac
                 has_tour          
Model %          0             66.8
                 1             33.2
Survey %         0             73.2
                 1             26.8
Difference (S-M) 0              6.4
                 1             -6.4

In [19]:
comparison.summary_day_pattern_purpose_rate("school")

psexpfac
                 has_tour          
Model %          0             85.6
                 1             14.4
Survey %         0             85.6
                 1             14.4
Difference (S-M) 0             -0.0
                 1              0.0

In [20]:
comparison.summary_day_pattern_purpose_rate("escort")

psexpfac
                 has_tour          
Model %          0             89.1
                 1             10.9
Survey %         0             93.7
                 1              6.3
Difference (S-M) 0              4.6
                 1             -4.6

In [21]:
comparison.summary_day_pattern_tour_count("work")
comparison.summary_day_pattern_tour_count("school")
comparison.summary_day_pattern_tour_count("escort")

psexpfac
                 wktopt          
Model %          0           66.8
                 1           30.7
                 2            2.4
                 3            0.1
Survey %         0           73.2
                 1           25.3
                 2            1.4
                 3            0.1
Difference (S-M) 0            6.4
                 1           -5.3
                 2           -1.0
                 3           -0.0

psexpfac
                 sctopt          
Model %          0           85.6
                 1           13.7
                 2            0.6
                 3            0.0
Survey %         0           85.6
                 1           13.8
                 2            0.6
                 3            0.0
Difference (S-M) 0           -0.0
                 1            0.1
                 2           -0.0
                 3           -0.0

psexpfac
                 estopt          
Model %          0           89.1
                 1            7.3
                 2            3.0
                 3            0.7
Survey %         0           93.7
                 1            4.6
                 2            1.4
                 3            0.2
Difference (S-M) 0            4.6
                 1           -2.7
                 2           -1.5
                 3           -0.4

survey is 100

In [22]:
comparison.summary_day_pattern_subtour_purpose_rate()

Work  School  Escort    PB  Shop  Meal  SocRec
                 has_subtour                                                
Model %          0            95.5   100.0    99.5  96.8  99.1  94.4    99.3
                 1             4.5     0.0     0.5   3.2   0.9   5.6     0.7
Survey %         0           100.0   100.0   100.0 100.0 100.0 100.0   100.0
                 1             0.0     0.0     0.0   0.0   0.0   0.0     0.0
Difference (S-M) 0             4.5     0.0     0.5   3.2   0.9   5.6     0.7
                 1            -4.5     0.0    -0.5  -3.2  -0.9  -5.6    -0.7

survey is NaN

In [23]:
comparison.summary_work_tour_mode()

psexpfac
                 tourmode                
Model %          Drive Alone         72.2
                 Shared Ride 2       17.2
                 Shared Ride 3+       9.9
                 Walk-Transit         0.3
                 Drive-Transit        0.3
                 Bike                 0.2
Survey %         Drive Alone          NaN
                 Shared Ride 2        NaN
                 Shared Ride 3+       NaN
                 Walk-Transit         NaN
                 Drive-Transit        NaN
                 Bike                 NaN
Difference (S-M) Drive Alone          NaN
                 Shared Ride 2        NaN
                 Shared Ride 3+       NaN
                 Walk-Transit         NaN
                 Drive-Transit        NaN
                 Bike                 NaN

In [24]:
comparison.summary_school_tour_mode()

psexpfac
                 tourmode                
Model %          School Bus          30.4
                 Walk-Transit         1.2
                 Shared Ride 2       23.5
                 Shared Ride 3+      33.2
                 Drive Alone          9.7
                 Bike                 0.1
                 Walk                 1.9
Survey %         School Bus           NaN
                 Walk-Transit         NaN
                 Shared Ride 2        NaN
                 Shared Ride 3+       NaN
                 Drive Alone          NaN
                 Bike                 NaN
                 Walk                 NaN
Difference (S-M) School Bus           NaN
                 Walk-Transit         NaN
                 Shared Ride 2        NaN
                 Shared Ride 3+       NaN
                 Drive Alone          NaN
                 Bike                 NaN
                 Walk                 NaN

In [25]:
comparison.summary_escort_tour_mode()

psexpfac
                 tourmode                
Model %          School Bus           0.0
                 Walk-Transit         0.0
                 Shared Ride 2+      94.2
                 Drive Alone          0.1
                 Bike                 1.9
                 Walk                 3.8
Survey %         School Bus           NaN
                 Walk-Transit         NaN
                 Shared Ride 2+       NaN
                 Drive Alone          NaN
                 Bike                 NaN
                 Walk                 NaN
Difference (S-M) School Bus           NaN
                 Walk-Transit         NaN
                 Shared Ride 2+       NaN
                 Drive Alone          NaN
                 Bike                 NaN
                 Walk                 NaN

arr_tod and dep_to have NaNs

In [26]:
comparison.summary_work_tour_arr_tod()
comparison.summary_work_tour_dep_tod()


Error [summary_work_tour_arr_tod]: cannot convert float NaN to integer
Error [summary_work_tour_dep_tod]: cannot convert float NaN to integer


arr_tod works but dep_tod and dur have NaN's

In [27]:
comparison.summary_school_tour_arr_tod()
comparison.summary_school_tour_dep_tod()
comparison.summary_school_tour_dur()

psexpfac
                 arrhour          
Model %          4             0.0
                 5             0.1
                 6            14.3
                 7            26.7
                 8            27.8
                 9            17.5
                 10            2.1
                 11            2.0
                 12            1.7
                 13            0.8
                 14            0.8
                 15            0.8
                 16            1.6
                 17            1.5
                 18            1.4
                 19            0.4
                 20            0.3
                 21            0.3
                 22            0.0
Survey %         4             0.0
                 5             0.2
                 6             8.2
                 7            48.6
                 8            19.6
                 9             8.0
                 10            2.7
                 11            1.1
                 12            1.4
                 13            2.4
                 14            1.5
                 15            2.2
                 16            0.7
                 17            2.0
                 18            0.9
                 19            0.3
                 20            0.0
                 21            0.0
                 22            0.0
Difference (S-M) 4            -0.0
                 5             0.1
                 6            -6.1
                 7            21.9
                 8            -8.2
                 9            -9.5
                 10            0.6
                 11           -0.8
                 12           -0.3
                 13            1.6
                 14            0.7
                 15            1.4
                 16           -0.9
                 17            0.5
                 18           -0.4
                 19           -0.1
                 20           -0.3
                 21           -0.3
                 22            0.0

psexpfac
                 depbin          
Model %          7-9          3.5
                 10-12        7.5
                 13-14       29.6
                 16          15.6
                 17           9.6
                 other       34.2
Survey %         7-9          6.6
                 10-12       10.0
                 13-14       29.7
                 16           9.1
                 17           5.1
                 other       39.4
Difference (S-M) 7-9          3.1
                 10-12        2.6
                 13-14        0.1
                 16          -6.5
                 17          -4.5
                 other        5.2

Error [summary_school_tour_dur]: cannot convert float NaN to integer


survey is NaN - trip mode not prepared correctly in rmd

In [28]:
comparison.summary_trip_mode_calib()

psexpfac
                 tripmode                       
Model %          Drive Alone                48.2
                 Shared Ride 2              26.9
                 Shared Ride 3+             17.3
                 Transit-local bus           0.5
                 Transit-light rail          0.0
                 Transit-premium bus         0.0
                 Transit-commuter rail       0.0
                 Transit-ferry               0.0
                 School Bus                  2.5
                 Bike                        0.5
                 Walk                        4.0
Survey %         Drive Alone                 NaN
                 Shared Ride 2               NaN
                 Shared Ride 3+              NaN
                 Transit-local bus           NaN
                 Transit-light rail          NaN
                 Transit-premium bus         NaN
                 Transit-commuter rail       NaN
                 Transit-ferry               NaN
                 School Bus                  NaN
                 Bike                        NaN
                 Walk                        NaN
Difference (S-M) Drive Alone                 NaN
                 Shared Ride 2               NaN
                 Shared Ride 3+              NaN
                 Transit-local bus           NaN
                 Transit-light rail          NaN
                 Transit-premium bus         NaN
                 Transit-commuter rail       NaN
                 Transit-ferry               NaN
                 School Bus                  NaN
                 Bike                        NaN
                 Walk                        NaN

# Other summaries

## Vehicle Availability

In [29]:
# by household incomeummary
#compare_summaries(model.summary_vehavail("inccat"),survey.summary_vehavail("inccat"))
comparison.summary_vehavail("inccat",pct_func=row_pct)

hhvehcat                    0     1     2     3     4
                 inccat                              
Model %          0K-15K  36.2  51.4  10.2   1.7   0.5
                 15K-50K  7.7  46.3  28.8  10.6   6.6
                 50K-75K  1.7  19.4  45.0  20.6  13.3
                 >75K     2.6  11.2  40.9  24.8  20.5
Survey %         0K-15K  29.8  47.6  14.0   6.5   2.1
                 15K-50K 10.9  49.1  27.9   9.0   3.1
                 50K-75K  2.7  38.4  34.1  17.1   7.6
                 >75K     0.0   0.0   0.0   0.0   0.0
Difference (S-M) 0K-15K  -6.4  -3.8   3.8   4.8   1.6
                 15K-50K  3.2   2.8  -0.9  -1.6  -3.5
                 50K-75K  1.0  19.0 -10.9  -3.5  -5.7
                 >75K    -2.6 -11.2 -40.9 -24.8 -20.5

In [30]:
# by county
comparison.summary_vehavail("hhcounty")




hhvehcat                      0     1     2     3     4
                 hhcounty                              
Model %          1         91.1  79.1  77.7  76.0  75.5
                 2          5.2  12.0  13.9  15.2  15.7
                 3          0.6   1.3   0.8   0.8   0.6
                 4          3.2   7.5   7.6   8.0   8.2
Survey %         1          0.0   0.0   0.0   0.0   0.0
                 2          0.0   0.0   0.0   0.0   0.0
                 3          0.0   0.0   0.0   0.0   0.0
                 4          0.0   0.0   0.0   0.0   0.0
Difference (S-M) 1        -91.1 -79.1 -77.7 -76.0 -75.5
                 2         -5.2 -12.0 -13.9 -15.2 -15.7
                 3         -0.6  -1.3  -0.8  -0.8  -0.6
                 4         -3.2  -7.5  -7.6  -8.0  -8.2

In [31]:
# by household drivers
#comparison.summary_vehavail("hh16cat")
compare_summaries(model.summary_vehavail("hh16cat"),survey.summary_vehavail("hh16cat"),pct_func=row_pct)

hhvehcat                     0    1    2    3    4
                 hh16cat                          
Model %          1        22.7 60.1 11.6  2.8  2.7
                 2         4.4 13.5 52.6 19.1 10.4
                 3         2.9  7.2 32.2 36.3 21.5
                 4         4.6  5.5  5.9 32.1 51.9
Survey %         1        11.5 69.7 13.5  4.5  0.8
                 2         3.1 15.1 56.7 18.0  7.1
                 3         0.8 12.7 24.9 43.0 18.6
                 4         5.9  7.6 16.7 26.3 43.5
Difference (S-M) 1       -11.2  9.6  1.9  1.7 -1.9
                 2        -1.3  1.6  4.0 -1.1 -3.3
                 3        -2.1  5.5 -7.3  6.8 -2.9
                 4         1.3  2.1 10.8 -5.9 -8.4

## Work/School Location

In [32]:
# work trip duration
# df = comparison.summary_wrkschloc_trip_duration("wrkr")
# df.columns = df.columns.astype(str)
# df = df.reset_index()
# #plt.bar(x=df['wrktimecat'],height=(df['FT']+df['PT']+df['NotFTPT']))

In [33]:
# work trip county flow
comparison.summary_wrkschloc_county_flow("wrkr")

pwcounty                   1.0   2.0   3.0   4.0   13.0
                 hhcounty                              
Model %          1         84.6  40.4  53.7  35.2  53.7
                 2          9.3  47.9   7.1  19.5  33.7
                 3          0.7   0.5  28.5   1.5   0.8
                 4          5.4  11.2  10.6  43.8  11.8
Survey %         1          0.0   0.0   0.0   0.0   0.0
                 2          0.0   0.0   0.0   0.0   0.0
                 3          0.0   0.0   0.0   0.0   0.0
                 4          0.0   0.0   0.0   0.0   0.0
Difference (S-M) 1        -84.6 -40.4 -53.7 -35.2 -53.7
                 2         -9.3 -47.9  -7.1 -19.5 -33.7
                 3         -0.7  -0.5 -28.5  -1.5  -0.8
                 4         -5.4 -11.2 -10.6 -43.8 -11.8

In [34]:
# work from home
comparison.summary_wrkschloc_at_home("wfh")

wrkrtyp                      FT    PT  NotFTPT
                 hhcounty                     
Model %          1         80.4  76.1     86.4
                 2         12.4  14.3      9.6
                 3          0.9   0.9      0.7
                 4          6.3   8.7      3.3
Survey %         1          0.0   0.0      0.0
                 2          0.0   0.0      0.0
                 3          0.0   0.0      0.0
                 4          0.0   0.0      0.0
Difference (S-M) 1        -80.4 -76.1    -86.4
                 2        -12.4 -14.3     -9.6
                 3         -0.9  -0.9     -0.7
                 4         -6.3  -8.7     -3.3

In [35]:
# school trip length
comparison.summary_wrkschloc_trip_length("stud")
#df.columns = df.columns.astype(str)
#df = df.reset_index()
#plt.bar(x=df['schdistcat'],height=(df['Ch515']+df['Stu16']+df['UniStu']))

stutyp                       Ch515  Stu16  UniStu
                 schdistcat                      
Model %          0.0          20.2    6.7    10.0
                 1.0          17.7    8.4     2.4
                 2.0          15.1   12.1     3.1
                 3.0          10.5   11.3     3.6
                 4.0           5.4   11.1     3.8
...                            ...    ...     ...
Difference (S-M) 41.0         -0.0    0.0     0.0
                 42.0         -0.0    0.0     0.0
                 43.0         -0.0    0.0    -0.0
                 45.0          0.0   -0.0     0.0
                 48.0          0.0    0.0    -0.0

[138 rows x 3 columns]

In [36]:
# school trip duration
# df = comparison.summary_wrkschloc_trip_duration("stud")
# df.columns = df.columns.astype(str)
# df = df.reset_index()
# plt.bar(x=df['schtimecat'],height=(df['Ch515']+df['Stu16']+df['UniStu']))

In [37]:
# school trip county flow
comparison.summary_wrkschloc_county_flow("stud")


pscounty                    1.0   2.0   3.0   4.0
                 hhcounty                        
Model %          1         86.9  15.0  56.3  14.2
                 2          8.2  76.4  11.5  14.9
                 3          0.5   0.1   9.8   1.2
                 4          4.4   8.6  22.4  69.7
Survey %         1          0.0   0.0   0.0   0.0
                 2          0.0   0.0   0.0   0.0
                 3          0.0   0.0   0.0   0.0
                 4          0.0   0.0   0.0   0.0
Difference (S-M) 1        -86.9 -15.0 -56.3 -14.2
                 2         -8.2 -76.4 -11.5 -14.9
                 3         -0.5  -0.1  -9.8  -1.2
                 4         -4.4  -8.6 -22.4 -69.7

In [38]:
# school at home
comparison.summary_wrkschloc_at_home("sfh")

stutyp                     Ch515  Stu16  UniStu
                 hhcounty                      
Model %          1          81.2   80.8    93.5
                 2          13.0   12.1     4.7
                 3           0.2    0.0     0.1
                 4           5.6    7.0     1.8
Survey %         1           0.0    0.0     0.0
                 2           0.0    0.0     0.0
                 3           0.0    0.0     0.0
                 4           0.0    0.0     0.0
Difference (S-M) 1         -81.2  -80.8   -93.5
                 2         -13.0  -12.1    -4.7
                 3          -0.2    0.0    -0.1
                 4          -5.6   -7.0    -1.8

## Trip Mode

In [39]:
# Work
comparison.summary_trip_mode(purpose=1) 
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased

Error [summary_trip_mode]: 'tripmode'


## Tour Mode

In [40]:
# Work
comparison.summary_tour_mode(purpose=1)
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased



vehcat                           0-Veh HHs  >0-Veh HHs
                 tourmode                             
Model %          Drive Alone           0.0        73.4
                 Shared Ride 2        51.3        16.2
                 Shared Ride 3+       28.5         9.3
                 Drive-Transit         0.0         0.3
                 Walk-Transit          4.1         0.2
                 Bike                  3.2         0.1
                 Walk                 12.9         0.5
                 School Bus            0.0         0.0
Survey %         Drive Alone           NaN         NaN
                 Shared Ride 2         NaN         NaN
                 Shared Ride 3+        NaN         NaN
                 Drive-Transit         NaN         NaN
                 Walk-Transit          NaN         NaN
                 Bike                  NaN         NaN
                 Walk                  NaN         NaN
                 School Bus            NaN         NaN
Difference (S-M) Drive Alone           NaN         NaN
                 Shared Ride 2         NaN         NaN
                 Shared Ride 3+        NaN         NaN
                 Drive-Transit         NaN         NaN
                 Walk-Transit          NaN         NaN
                 Bike                  NaN         NaN
                 Walk                  NaN         NaN
                 School Bus            NaN         NaN

## Trip Destination

In [41]:
# trip length by trip purpose
comparison.summary_trip_destination("distcat").reset_index()
#plt.bar(x=df['distcat'],height=df[0]) 
# 0-non/home  1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 10-change mode inserted purpose



Error [summary_trip_destination_distca]: 'Column not found: psexpfac'


AttributeError: 'NoneType' object has no attribute 'reset_index'

In [ ]:
# trip duration by trip purpose
comparison.summary_trip_destination("timecat").reset_index()
#plt.bar(x=df['timecat'],height=df[0])
# 0-non/home  1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 10-change mode inserted purpose



## Tour Destination

In [ ]:
# tour length by tour purpose
comparison.summary_tour_destination("distcat").reset_index()
#plt.bar(x=df['distcat'],height=df[1])
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased

In [ ]:
# tour duration by tour purpose
comparison.summary_tour_destination("timecat").reset_index()
#plt.bar(x=df['timecat'],height=df[1])
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased

In [ ]:
# tour county flow by tour purpose
comparison.summary_tour_destination_county_flow(purpose=1)
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased


## Trip Time of Day

In [ ]:
# Trip arrival time at stop location - outbound (first) half-tour
trip_tod = compare_summaries(
    model.summary_trip_tod("arrtimecat", filter_by_var="arrflag"),
    survey.summary_trip_tod("arrtimecat", filter_by_var="arrflag")
)
tables["trip_tod_arrflag"] = trip_tod
display(trip_tod)
df = trip_tod.reset_index()
plt.bar(x=df['arrtimecat'],height=df[1])
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 10-change mode inserted purpose




In [ ]:
# Trip departure time at stop location - return (second) half-tour
comparison.summary_trip_tod("deptimecat",filter_by_var="depflag").reset_index()
#plt.bar(x=df['deptimecat'],height=df[1])
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 10-change mode inserted purpose



In [ ]:
# Duration at stop location - both half-tours
comparison.summary_trip_tod("durdestcat",filter_by_var="durflag").reset_index()
#plt.bar(x=df['durdestcat'],height=df[1])
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 10-change mode inserted purpose



In [ ]:
# Trip arrival time All
comparison.summary_trip_tod("arrtimecat",filter_by_var=False).reset_index()
#plt.bar
#(x=df['arrtimecat'],height=df[1])
## 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 10-change mode inserted purpose



In [ ]:
# Trip departure time All
comparison.summary_trip_tod("deptimecat",filter_by_var=False).reset_index()
#plt.bar(x=df['deptimecat'],height=df[1])
# 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 10-change mode inserted purpose

## Tour Time of Day

In [ ]:
# arrival time all trip purposes
comparison.summary_tour_tod("arrtimecat").reset_index()
#plt.bar(x=df['arrtimecat'],height=df[1]) 
# 1 Work 2 School 3 Other 4 Workbased



In [ ]:
# departure time all trip purposes
comparison.summary_tour_tod("deptimecat").reset_index()
#plt.bar(x=df['deptimecat'],height=df[1]) 
# 1 Work 2 School 3 Other 4 Workbased

In [ ]:
# duration all trip purposes
comparison.summary_tour_tod("durdestcat").reset_index()
#plt.bar(x=df['durdestcat'],height=df[2]) 
# 1 Work 2 School 3 Other 4 Workbased



In [ ]:
# arrival time by trip purpose by person type
comparison.summary_tour_tod_purpose("arrtimecat", purpose=2).reset_index()
# purpose: 1 Work 2 School 3 Other 4 Workbased
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5


In [ ]:
# departure time by trip purpose by person type
comparison.summary_tour_tod_purpose("deptimecat", purpose=2).reset_index()
# purpose: 1 Work 2 School 3 Other 4 Workbased
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5



## Person Day Pattern

In [ ]:
# number of tours by person type
comparison.summary_day_pattern_num_of_tours()
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5



In [ ]:
# tour/stop combinations by person type
comparison.summary_day_pattern_tour_stops()
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5
# index: tours/stops
# 0 0/0
# 1 1/0
# 2 1/1
# 3 1/2
# 4 1/3+
# 5 2/0
# 6 2/1
# 7 2/2
# 8 2/3+
# 9 3+/0
# 10 3+/1
# 11 3+/2
# 12 3+/3+

In [ ]:
# tour/stop combinations by purpose by person type
comparison.summary_day_pattern_tour_stops_by_purpose(purpose="wktostp")
# purpose: "wktostp", "sctostp", "estostp", "pbtostp", "shtostp", "mlstops", "sotostp"
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5
# index: tours/stops
# 1 0/0
# 2 0/1+
# 3 1+/0
# 4 1+/1+



In [ ]:
# tours by purpose by person type
comparison.summary_day_pattern_tours_by_purpose(purpose="wktopt")
# purpose: "wktopt", "sctopt", "estopt", "pbtopt", "shtopt", "mltopt", "sotopt"
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5
# index: 0,1,2,3+



In [ ]:
# number of subtours (work tours only)
comparison.summary_day_pattern_subtours()
# columns: 1-FT 2-Other
# index: 0,1,2,3+



In [ ]:
# number of subtours (work tours only) by purpose
comparison.summary_day_pattern_subtours_by_purpose()
# columns: 1-FT 2-Other
# index :1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational



In [ ]:
# estimated tours by number of stops and purpose
comparison.summary_day_pattern_stops_by_tour_purpose("stopscat")
# columns: 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational
# index: 0,1,2,3,4,5,6+



In [ ]:
# estimated outbound tours by number of stops and purpose
comparison.summary_day_pattern_stops_by_tour_purpose("h1stopscat")
# columns: 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational
# index: 0,1,2,3,4,5,6+



In [ ]:
# estimated return tours by number of stops and purpose
comparison.summary_day_pattern_stops_by_tour_purpose("h2stopscat")
# columns: 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational
# index: 0,1,2,3,4,5,6+



In [ ]:
# estimated total tours by purpose by person type
comparison.summary_day_pattern_tours_by_tour_purpose("pptyp")
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5
# index: 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased



In [ ]:
# estimated total tours by purpose by household income
comparison.summary_day_pattern_tours_by_tour_purpose("inccat")
# index: 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased



In [ ]:
# estimated total tours by purpose by auto sufficiency
comparison.summary_day_pattern_tours_by_tour_purpose("vehsuf")
# columns: 1-0, 2-autos<drivers, 3-autos=drivers, 4-autos>drivers
# index: 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased



In [ ]:
# estimated total tours by purpose by county
comparison.summary_day_pattern_tours_by_tour_purpose("hhcounty")
# index: 1-Work 2-School 3-Escort 4-Personal_Business 5-Shop 6-Meal 7-Social&Recreational 8-Workbased



In [ ]:
# estimated total stops by purpose by person type
comparison.summary_day_pattern_stops_by_stop_purpose_agg("pptyp")
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5

In [ ]:
# estimated total stops by purpose by household income
comparison.summary_day_pattern_stops_by_stop_purpose_agg("inccat")


In [ ]:
# estimated total stops by purpose by auto sufficiency
comparison.summary_day_pattern_stops_by_stop_purpose_agg("vehsuf")
# columns: 1-0, 2-autos<drivers, 3-autos=drivers, 4-autos>drivers



In [ ]:
# estimated total stops by purpose by county
comparison.summary_day_pattern_stops_by_stop_purpose_agg("hhcounty")

In [ ]:
# estimated total trips by purpose by person type
comparison.summary_day_pattern_trips_by_destination_purpose("pptyp")
# columns: 1-FT 2-PT 3-Retired 4-Nonworker 5-UnivStud 6-Stud16+ 7-Stud5-15 8-Child<5




In [ ]:
# estimated total trips by purpose by household income
comparison.summary_day_pattern_trips_by_destination_purpose("inccat")



In [ ]:
# estimated total trips by purpose by auto sufficiency
comparison.summary_day_pattern_trips_by_destination_purpose("vehsuf")
# columns: 1-0, 2-autos<drivers, 3-autos=drivers, 4-autos>drivers



In [ ]:
# estimated total trips by purpose by county
comparison.summary_day_pattern_trips_by_destination_purpose("ocounty")




In [ ]:
output_file = r"C:\Users\USVA682771\OneDrive - WSP O365\Documents\client_chattanooga_TNHTS\data\processed\survey_daysim\DaySim_Survey_Comparisons.xlsx"
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for sheet_name, df in tables.items():
        df.to_excel(writer, sheet_name=sheet_name[:31])  # Excel sheet names have a max length of 31 characters

In [ ]:
rows = []
for key, df in tables.items():
    long = (df["Model %"].stack().rename("Model %")
            .to_frame()
            .join(df["Survey %"].stack().rename("Survey %"))
            .join(df["Difference (S-M)"].stack().rename("Difference (S-M)"))
            .reset_index())
    long.insert(0, 'Comparison', key)
    rows.append(long)
long_df = pd.concat(rows, ignore_index=True)
long_df.to_csv('model_survey_comparisons.csv', index=False)

In [ ]:
errors
